# Trabajo final · Data Analyst · Casa Ceibo

**Hotel boutique** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real de ARCA.

## La pregunta oficial

> **¿Conviene bajar las tarifas en los meses bajos para llenar el hotel?**

**Decisión que habilita:** Si se extiende, se ajusta o se retira la tarifa promocional.

Antes de analizar, **verificá si los datos sostienen la premisa de la pregunta**. A veces no la sostienen, y decirlo es un hallazgo.

Las instrucciones completas (qué se entrega, rúbrica y defensa) están en el PDF **«Trabajo final · Data Analyst · Instrucciones»**. Este notebook es tu lugar de trabajo: vas del **SQL al análisis** sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 2 | SQL y PL/SQL |
| 3 | Estadística |
| 4 | Python: limpieza |
| 5 | Proceso de análisis |
| 6 | Superset |
| 7 | Analítica de negocio |
| 8 | Bitácora y entrega |

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib"}
print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")

## 1 · La base de Casa Ceibo

La primera celda **genera la base** con código comprimido a propósito: no hace falta leerla, y leerla te quita la mitad del trabajo. La segunda la carga en SQLite y te deja tres herramientas: `sql(...)` para consultar, `esquema()` para ver las tablas y `anotar(...)` para dejar registrado cada problema de calidad que encuentres.

In [ ]:
import zlib, base64
_BLOB = (
    "eNp9VuGOozYQ/r9P4SKdBLesBUnYy2abH5Xaqv8qtatKFUKRF5yFW7CRTXLdrvZh+ix9sX5jSIDsXVEUjD0znvm+mbGrptWm"
    "Y+rQtC9MWKbakLVCFRji1xZXhdyznT3klfZNyKw0lQzZUZhKqE5aCAebK4bHsm2/ynPdvvgBF7Z7aaWvHz/LvAvuWQMBww1s"
    "68avpfJtELDvWdtrp02G9fRs2D+GzARsrw07sko5geyeGdkdDL6uer8a8XKwO2lbQf45lc1JJj3yWn+Rxg/gLz+07TBk18xj"
    "Xkh/GB6z1PAKOz5JY/0oZKsg640XItnlwopdLqtH7Vspi20SJUO4Bt6qdgiHQ14c6m5n1JMTRLhd1WrCJPW9n2z37z+A1GDX"
    "RcjukijCRvEC3vje7wc4VmlaW9Jkv7ge1qpOYmGFhbvILcC9e1aKR7KcOU8Ioi5kuQBzRuTPIVMlIeYc6J09ie1oHi4/SV+V"
    "QAp2uAAwqvD9vffLK7GCuQC4xJtoUbxh76npIHDm+u3bgv8oOvGzEY0kLcjp+tAou029qthhpuqIFkVYkzP0hinMFaJwkyB7"
    "L3Zk2csIMtm0MPwab5gn6k4QXOPw0ziMJ9NLDBtZVDReTcZxPPlIMH4Un2l4Ow7X4/BuHMbRafzmoq0FZe5rv9+GfeKgaTC8"
    "YYn7ctIbtuR3b/dsL/KpfMwXyUQh5tGowNcJFOrq0UhSKTeUUn9Lo62/IrIfta77GnCMEl0zZGm3qhYuzZAWz3hH95Csenoe"
    "qkbaTjSt7y2iRXITxfh5wTlr4NGYEMvbU2rTU8AC2bk+2SkkwvEL8WK3UANZtCkRlha80aorsX9rdKMxPcyQbR+x30JaueJv"
    "dWWtVj4gTW3GPjIfcKxYtR9UZW0li4cs+1rSTjykh7LKGc5LXeXST2elNi2toZQy9KttyuEUX9JfnFDeKZ2X0rqIO380Frtq"
    "Xbryg/CnQTlekzZW+GJFJmgU3dEfjMFaLpSo514VlUEPdAXw68MP9BJPUuVIiMHmCmTz1egRbBADaTkjP6UE6GPeutizeToQ"
    "kEp3fUKlZZaCqQ2RfD2EmHGh0JqzGYiDEm25QQmrrlIHOZMo4QotpxFIrqUoBqiQrHldtYCs1k9Km0bU/pLym9+hfeG9jCJC"
    "BEk4xwO77CuIF6LvCSqX9fCh9M6W+ksPzOsZuQ1L+dqBTOhHy2yAkuZvaX4xzp+wdTpLwnTVr72ljpssuIyfHPxuO/Vk8z8g"
    "IpYHcwER9TDM72stOmqGQCRP39Ur8VZeNj5ekWzkGqCgY8XoAxqys/iRuslQKnwdvasUEnBbGj7gH1OoqwD43ywCagnXWxbP"
    "XHUdY9L2f/vj9XkTJa7Zwzm/YDdfKXqiPQi47cy+w4rvffjz5kNz86HwsFXxjfkesLAviZDCC9mU0+jMoquqCXM8jk5shUNZ"
    "ng9phJhQgGBtaBXmeHkeuSAvTiQjcT85uix7dz7tZV6KqQQcz593lepzksIYUlXUk6PLrbgF3VQWtnZt3tF3ecClBPg5LWrC"
    "hXZH3LtLkDm6WxCPoiVuN0eXN83lBuAfLEZngW9YWIy+UpJGPTaus5ljCt3eEcrCaQlm3OKMqAEacm7Lk4QOezK/g3gnt+5C"
    "k84gzLLhlkc3Mpt67UF14rN02566At2O+kw+5eaa00WGgz/nt8QFEDTGdOSeDM2JyZzn3MoOTBTyL3/qQ9BDIS0fJ99Rm/Gj"
    "qMHF/TlHEHiOggEcobPdh04N+H3QlHvU2GS//ZbKPpihFX9NjZM3J5cLo9tekXxIhxSiyMZbNfwYSmR+o3Vta0jx/lL76o3B"
    "gWZ3hUPQQ/g0QXG5b9V/AtXL7D+zlb1dXaHkpHF9Z37hvfoPTPihyQ=="
)
exec(zlib.decompress(base64.b64decode(_BLOB)).decode("utf-8"))   # define generar(): código comprimido a propósito

In [ ]:
import sqlite3, matplotlib.pyplot as plt
pd.options.display.width = 200; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.3f}".format
t = generar(); tablas = {k: v for k, v in t.items() if not k.startswith("_")}
con = sqlite3.connect(":memory:")
for nombre, df in tablas.items(): df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False)

def sql(consulta, parametros=None):
    """Corre una consulta SQL sobre la base y devuelve un DataFrame."""
    return pd.read_sql_query(consulta, con, params=parametros)

def esquema():
    """Lista las tablas, sus columnas y sus filas."""
    for nombre, df in tablas.items(): print(f"{nombre}  ({len(df):,} filas)".replace(",", "."), "\n   ", ", ".join(df.columns))

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset"):
    """Guarda la base como archivo SQLite y un CSV por tabla, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); salida = sqlite3.connect(f"{carpeta}/base.db")
    for nombre, df in tablas.items(): df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, salida, index=False, if_exists="replace")
    salida.close(); print(f"Listo: {carpeta}/base.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta
import os
esquema()

Los datos tienen los problemas de un dato real. Encontrarlos, decidir qué hacer con cada uno y dejarlo escrito es parte de la evidencia.

## 2 · SQL y PL/SQL

Tres consultas. Guardalas en esta sección: son la evidencia del módulo.

1. La ocupación diaria o mensual: hay que repartir las noches de cada reserva (una `CTE` recursiva o una tabla de fechas).
2. La cancelación y el ingreso neto por canal.
3. El RevPAR por mes.

SQLite no tiene PL/SQL. Si querés encapsular una lógica, escribila como una función de Python que ejecuta SQL.

In [ ]:
sql("""
-- Consulta 1
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 2
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 3 (control de integridad)
SELECT 1
""")

## 3 · Estadística

Para la variable principal (`tarifa_noche`): **media, mediana, dispersión y distribución**. Decidí cuál resume mejor y escribí por qué.

In [ ]:
# Tu análisis estadístico

## 4 · Python: limpieza

Traé las tablas a pandas (`df = sql("SELECT * FROM tabla")`) y documentá la calidad: **faltantes, duplicados, valores imposibles y categorías inconsistentes**. Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)`.

In [ ]:
# Tu limpieza
# anotar("problema", "tabla.columna", 0, "qué decidimos y por qué")

In [ ]:
bitacora_calidad

## 5 · Proceso de análisis

**Verificá la premisa** de la pregunta oficial antes de seguir. Después armá al menos dos variables nuevas (estacionalidad, anticipación de la reserva, noches ocupadas, ingreso neto de comisión).

In [ ]:
# Tu verificación de la premisa y tus variables nuevas

## 6 · Dashboard en Superset

Exportá la base y conectá Apache Superset al archivo. Armá **4 a 6 gráficos, uno por pregunta**, con un orden de lectura y las definiciones a la vista. Si no podés usar Superset, trabajá con gráficos de este notebook.

In [ ]:
exportar_superset()

In [ ]:
# Si no usás Superset, tus gráficos con matplotlib

## 7 · Analítica de negocio

**KPIs** con su denominador: ocupación (noches ocupadas sobre noches disponibles), tarifa media por noche (ADR), ingreso por habitación disponible (RevPAR), cancelación por canal e ingreso neto de comisión.

**Segmentación:** canal, tipo de habitación y mes.

Un **insight** relevante, una **limitación** y una **recomendación** proporcional a la evidencia.

In [ ]:
# Tus KPIs y tu segmentación

## 8 · Bitácora y entrega

Completá esta bitácora editando la celda. Con ella, otra persona tiene que poder rehacer tus números.

**Equipo:** ____  **Integrantes y roles:** ____

**Pregunta y premisa:** ¿los datos sostienen la premisa? ____

| KPI | Fórmula | Universo | Período | Estados o faltantes |
| --- | --- | --- | --- | --- |
|  |  |  |  |  |

**Hechos** (lo que muestran los datos): ____

**Inferencias** (lo que creemos que significa): ____

**Lo que contradice nuestra hipótesis:** ____

**Recomendación** (proporcional a la evidencia): ____

**Límites y siguiente análisis:** ____

### Antes de entregar

- [ ] Reinicié el kernel y corrí todo el notebook de punta a punta.
- [ ] Mi pregunta termina en una decisión y nombra quién la toma.
- [ ] Verifiqué la premisa de la pregunta.
- [ ] Cada KPI declara fórmula, universo, período y tratamiento de estados.
- [ ] Ningún porcentaje se muestra sin decir sobre cuántos casos.
- [ ] La limpieza está anotada: qué encontré, qué hice y cuántas filas afectó.
- [ ] Los hechos están separados de las inferencias y no prometo causalidad.
- [ ] Mostré al menos un resultado que no me conviene.
- [ ] Hay una limitación y un siguiente análisis.